In [ ]:
!pip install unsloth "xformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 56.4 MB/s e

In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import random

In [ ]:
import torch
import gc


In [ ]:
import pandas as pd

In [ ]:
import torch.nn.functional as F
import numpy as np

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def generate_seeds(num_seeds=20, seed=42):
    """Generates a list of random seeds.

    Args:
        num_seeds: The number of seeds to generate.
        seed: The initial seed for the random number generator (for reproducibility).

    Returns:
        A list of random integer seeds.
    """
    random.seed(seed)  # Set initial seed for reproducibility
    seeds = [random.randint(1, 100000) for _ in range(num_seeds)]
    return seeds

In [ ]:
def create_text_generation_pipeline(model, tokenizer, temperature=1.0):
    """
    Creates a text-generation pipeline with the given model and tokenizer.

    Args:
        model: The preloaded model for text generation.
        tokenizer: The corresponding tokenizer.
        temperature (float): Sampling temperature for generation (default: 1.0).
        max_new_tokens (int): Maximum number of tokens to generate (default: 1024).

    Returns:
        A transformers pipeline object for text generation.
    """
    return transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            trust_remote_code=True,
            pad_token_id=0,
            do_sample=True,
            temperature=1.0,
            max_new_tokens=1,
    )

# Example usage:
# pipe = create_text_generation_pipeline(model, tokenizer)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "marcelbinz/Llama-3.1-Centaur-8B-adapter",
        max_seq_length = 32768,
        dtype = None,
        load_in_4bit = True,
      )

==((====))==  Unsloth 2025.6.8: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

Unsloth 2025.6.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
pipe = create_text_generation_pipeline(model, tokenizer)

Device set to use cuda:0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
test_data=pd.read_csv('/content/drive/MyDrive/centaur/reversal_learning/rw_reversal_learning.csv')

In [ ]:
test_data['choice'] = test_data['choice'].map({0: 'U', 1: 'P'})

In [ ]:
def generate(prompt, pipe):
    # Convert the prompt list to a single string
    prompt_items = [str(item) if not isinstance(item, str) else item for item in prompt]
    prompt_str = "".join(prompt_items)
    return pipe(prompt_str)[0]['generated_text'][len(prompt_str):]

In [ ]:
def format_past_trials(past_df):
    """Format past free-choice trials for the prompt."""
    trials_text = []
    for _, row in past_df.iterrows():
        trials_text.append(f"You press <<{row['choice']}>> and get {row['reward']} points.")
    return trials_text

In [ ]:
def build_game_intro(trial_length):
    """Build the introduction part of the prompt for a game."""
    #total_trials = timeline_df['trial_num_block'].max()

    intro = [
             "In this task, you have to repeatedly choose between two slot machines labeled U and P.\n"
              "You can choose a slot machine by pressing its corresponding key."
              "When you select one of the machines, you will win 1 or 0 points."
              "Your goal is to choose the slot machines that will give you the most points."
              "You will receive feedback about the outcome after making a choice.\n"
              "The environment may change unpredictably, and past success does not guarantee future results. You’ll need to adapt to these changes to keep finding the better machine."
              f"You will play 1 game in total, consisting of {trial_length} trials."
            f" Game 1. There are {trial_length} trials in this game."
        ]
    return intro



In [ ]:
participants_in_test=len(test_data['model_id'].unique())

In [ ]:
seeds=generate_seeds(num_seeds=participants_in_test, seed=42)

In [ ]:
# Set up your tokenizer-to-token-ID mapping for decoding model output
letter_token_ids = {
    "U": tokenizer("U", add_special_tokens=False)['input_ids'][0],
    "P": tokenizer("P", add_special_tokens=False)['input_ids'][0],
}

In [ ]:
def simulate_centaur_on_human_data(pipe, df, model=None, tokenizer=None):
    """
    Replays human trials and logs Centaur's predicted choices, logits, and log-likelihoods
    without affecting outcomes. Model is never rewarded — just evaluated on human timeline.
    """
    history = []                         # what we log and return
    cumulative_reward = 0
    total_trials = df["trial"].max()   # or just len(df)

    for trial in range(1, total_trials + 1):
        past = df[df["trial"] < trial]

        # --- Build prompt based on past (not model history) ---
        game_intro = build_game_intro(total_trials)
        prompt = game_intro + format_past_trials(past) + ["You press <<"]
        prompt_str = ''.join(prompt)
        # --- Run model on prompt ---
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1]
        probs = F.softmax(logits, dim=-1)
        pred_token_id = torch.argmax(probs).item()

        # --- Map to model choice (U or P) ---
        token_id_to_letter = {v: k for k, v in letter_token_ids.items()}
        model_choice = token_id_to_letter.get(pred_token_id, "INVALID")

        # --- Log-likelihood of human’s actual choice ---
        user_choice = df.loc[df["trial"] == trial, "choice"].values[0]
        log_likelihood = None
        if user_choice in letter_token_ids:
            user_token_id = letter_token_ids[user_choice]
            log_likelihood = torch.log(probs[user_token_id] + 1e-8).item()

        # --- Get the actual reward (from human data) ---
        reward = df.loc[df["trial"] == trial, "reward"].values[0]
        cumulative_reward += reward

        # --- Save everything ---
        history.append({
            "trial_num": trial,
            "prompt": prompt_str,
            "model_choice": model_choice,
            "model_pred_token_id": pred_token_id,
            #"logits": logits.to(torch.float32).cpu().numpy().tolist(),
            #"probs": probs.to(torch.float32).cpu().numpy().tolist(),
            "log_likelihood_of_human_choice": log_likelihood,
            "human_choice": user_choice,
            "reward": reward,
            "cumulative_reward": cumulative_reward,
        })

        print(f"Trial {trial}: Human {user_choice}, Model {model_choice}, Reward {reward}")

    return history, cumulative_reward


In [ ]:
def fix_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    transformers.set_seed(seed)  # For Hugging Face models

In [ ]:
participant_ids = test_data['model_id'].unique().tolist()

In [ ]:
fix_seed(seeds[0])

In [ ]:
# Storage for results
all_results = []

# Run simulation for each seed
# Run simulation for each seed
for participant_id in participant_ids:
    df_participant=test_data[test_data['model_id']==participant_id]
    history = []
    cumulative_reward = 0
    total_trials = df_participant["trial"].max()
    print(f"\n### Running simulation with (Run {participant_id}/{len(participant_ids)}) ###\n")
    history, cumulative_reward = simulate_centaur_on_human_data(pipe, df_participant, model, tokenizer)
    all_results.append({
        "run_id": participant_id,
        "history": history,
        "final_score": cumulative_reward
    })



### Running simulation with (Run 83811/32) ###

Trial 1: Human U, Model U, Reward 1
Trial 2: Human P, Model P, Reward 0
Trial 3: Human U, Model U, Reward 1
Trial 4: Human P, Model U, Reward 1
Trial 5: Human U, Model U, Reward 1
Trial 6: Human U, Model P, Reward 1
Trial 7: Human U, Model P, Reward 1
Trial 8: Human P, Model U, Reward 0
Trial 9: Human U, Model U, Reward 1
Trial 10: Human U, Model U, Reward 0
Trial 11: Human U, Model P, Reward 1
Trial 12: Human P, Model U, Reward 1
Trial 13: Human P, Model P, Reward 0
Trial 14: Human P, Model U, Reward 0
Trial 15: Human U, Model U, Reward 1
Trial 16: Human P, Model U, Reward 0
Trial 17: Human U, Model U, Reward 1
Trial 18: Human P, Model U, Reward 0
Trial 19: Human U, Model U, Reward 1
Trial 20: Human U, Model U, Reward 1
Trial 21: Human U, Model U, Reward 1
Trial 22: Human U, Model U, Reward 1
Trial 23: Human U, Model U, Reward 1
Trial 24: Human U, Model U, Reward 1
Trial 25: Human U, Model U, Reward 1
Trial 26: Human U, Model U, Reward 

In [ ]:
all_results

[{'run_id': 83811,
  'history': [{'trial_num': 1,
    'prompt': 'In this task, you have to repeatedly choose between two slot machines labeled U and P.\nYou can choose a slot machine by pressing its corresponding key.When you select one of the machines, you will win 1 or 0 points.Your goal is to choose the slot machines that will give you the most points.You will receive feedback about the outcome after making a choice.\nThe environment may change unpredictably, and past success does not guarantee future results. You’ll need to adapt to these changes to keep finding the better machine.You will play 1 game in total, consisting of 100 trials. Game 1. There are 100 trials in this game.You press <<',
    'model_choice': 'U',
    'model_pred_token_id': 52,
    'log_likelihood_of_human_choice': -0.6328125,
    'human_choice': 'U',
    'reward': np.int64(1),
    'cumulative_reward': np.int64(1)},
   {'trial_num': 2,
    'prompt': 'In this task, you have to repeatedly choose between two slot m

In [ ]:
import pandas as pd

# Create an empty DataFrame
history_pd = pd.DataFrame(columns=['trial', 'choice', 'reward', 'score'])

# Initialize score
cumulative_score = 0

# Iterate through the history list
for i, trial_data in enumerate(history):
    # Calculate cumulative score
    cumulative_score += trial_data['reward']

    # Append data to the DataFrame
    history_pd.loc[len(history_pd)] = [i + 1, trial_data['choice'], trial_data['reward'],cumulative_score]

KeyError: 'choice'

In [ ]:
# Assuming all_results is a list of dictionaries
all_dfs = []
model_id=seeds[0]

for i in range(len(all_results)):  # Iterate through each model run
    #model_id = all_results[i]['seed']  # Retrieve model_id (seed)
    history = all_results[i]['history']  # Retrieve history

    df = pd.DataFrame(history)  # Convert history to DataFrame
    df['model_id'] = model_id

    df.rename(columns={'cumulative_reward': 'score'}, inplace=True)  # Rename column

    all_dfs.append(df)  # Append to list

# Concatenate all DataFrames into one
final_df = pd.concat(all_dfs, ignore_index=True)

print(final_df)


In [ ]:
final_df.to_csv('/content/drive/MyDrive/centaur/predictive_centaur_reversal_learning.csv')

In [ ]:
# prompt: sum log likelihoods and average them

# Assuming final_df is the DataFrame containing the log_likelihoods
# Filter out rows where log_likelihood_of_human_choice is None
#filtered_df = final_df.dropna(subset=['log_likelihood_of_human_choice'])

# Calculate the sum of log likelihoods
#sum_log_likelihoods = filtered_df['log_likelihood_of_human_choice'].sum()

# Calculate the number of valid log likelihoods
#count_log_likelihoods = filtered_df['log_likelihood_of_human_choice'].count()

# Calculate the average log likelihood
#average_log_likelihood = sum_log_likelihoods / count_log_likelihoods if count_log_likelihoods > 0 else 0

#print(f"Sum of log likelihoods: {sum_log_likelihoods}")
#print(f"Average log likelihood: {average_log_likelihood}")
